# Data Verzameling & Verkenning (EDA)
**Groep 2 | Hogeschool van Amsterdam | 2024-2025**

Dit notebook heeft twee doelen:
1. **Data verzameling** — controleren of de API-URLs bereikbaar zijn en data teruggeven
2. **EDA (Exploratory Data Analysis)** — de data bekijken, testen en opschonen

---

## 0. Bibliotheken importeren

We importeren de Python-pakketten die we nodig hebben.
- `requests` → voor het ophalen van data via APIs
- `pandas` → voor het werken met tabellen (DataFrames)
- `numpy` → voor wiskundige berekeningen
- `matplotlib` / `seaborn` → voor grafieken

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

# mooiere grafieken
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')

print('Alle bibliotheken geladen ✓')

---
## 1. DATA VERZAMELING — API-URLs controleren

We gebruiken twee APIs:
| # | Bron | URL |
|---|------|-----|
| 1 | CBS Kerncijfers Wijken & Buurten | `https://opendata.cbs.nl/ODataApi/odata/85984NED/TypedDataSet` |
| 2 | DUO Schooladviezen (wpoadvies-v1) | `https://onderwijsdata.duo.nl/api/3/action/package_show` |

Voor elke API controleren we:
- Reageert de server? (HTTP-statuscode)
- Hoe snel? (responstijd)
- Geeft de server data terug? (eerste paar rijen)

In [ ]:
def test_api(naam, url, params=None, verwacht_veld=None):
    """Test een API-URL en rapporteer de bevindingen."""
    print(f"\n{'='*60}")
    print(f"API: {naam}")
    print(f"URL: {url}")
    if params:
        print(f"Parameters: {params}")
    print()

    start = time.time()
    try:
        antwoord = requests.get(url, params=params, timeout=20)
        duur = round(time.time() - start, 2)

        # HTTP-statuscode
        code = antwoord.status_code
        if code == 200:
            print(f"✅ HTTP {code} — server bereikbaar (in {duur}s)")
        elif code in (301, 302):
            print(f"↪️  HTTP {code} — doorverwijzing (in {duur}s)")
        else:
            print(f"❌ HTTP {code} — fout! (in {duur}s)")
            return None

        # inhoud controleren
        try:
            data = antwoord.json()
            if verwacht_veld and verwacht_veld in data:
                items = data[verwacht_veld]
                print(f"✅ JSON ontvangen — '{verwacht_veld}' bevat {len(items)} rijen")
                if len(items) > 0 and isinstance(items[0], dict):
                    print(f"   Kolommen: {list(items[0].keys())[:8]} ...")
            else:
                sleutels = list(data.keys()) if isinstance(data, dict) else type(data).__name__
                print(f"✅ JSON ontvangen — sleutels: {sleutels}")
            return data
        except Exception:
            print(f"⚠️  Antwoord is geen JSON (lengte: {len(antwoord.text)} tekens)")
            return antwoord.text

    except requests.exceptions.Timeout:
        print(f"❌ TIMEOUT — server reageert niet binnen 20 seconden")
        return None
    except requests.exceptions.ConnectionError as e:
        print(f"❌ VERBINDINGSFOUT — {e}")
        return None
    except Exception as e:
        print(f"❌ ONBEKENDE FOUT — {e}")
        return None

### 1.1 CBS OData v3 API testen

In [ ]:
cbs_url    = "https://opendata.cbs.nl/ODataApi/odata/85984NED/TypedDataSet"
cbs_params = {
    "$filter": "startswith(WijkenEnBuurten,'WK0363')",  # alleen Amsterdam-wijken
    "$format": "json",
    "$top": 5   # maar 5 rijen om snel te testen
}

cbs_resultaat = test_api("CBS Kerncijfers Wijken & Buurten 2024", cbs_url, cbs_params, "value")

In [ ]:
# CBS: bekijk de eerste rij als tabel
if cbs_resultaat and 'value' in cbs_resultaat and len(cbs_resultaat['value']) > 0:
    cbs_sample = pd.DataFrame(cbs_resultaat['value'])
    print(f"Aantal rijen teruggegeven: {len(cbs_sample)}")
    print(f"Aantal kolommen: {len(cbs_sample.columns)}")
    print("\nEerste rij (geselecteerde kolommen):")
    
    # toon de meest relevante kolommen
    interessant = [c for c in cbs_sample.columns
                   if any(w in c.lower() for w in ['wijken', 'inwoners', 'inkomen', 'woz', 'bijstand', 'hbo', 'basis'])]
    display(cbs_sample[interessant].head(3))
else:
    print("⚠️ Geen CBS-data ontvangen — we gebruiken eigen nooddata")

### 1.2 DUO Open Onderwijsdata API testen

In [ ]:
duo_url    = "https://onderwijsdata.duo.nl/api/3/action/package_show"
duo_params = {"id": "wpoadvies-v1"}

duo_resultaat = test_api("DUO Schooladviezen (wpoadvies-v1)", duo_url, duo_params)

In [ ]:
# DUO: welke bestanden zijn beschikbaar?
if duo_resultaat and isinstance(duo_resultaat, dict):
    package = duo_resultaat.get('result', {})
    bronnen = package.get('resources', [])
    print(f"Beschrijving dataset: {package.get('notes', 'geen')[:200]}")
    print(f"\nAantal beschikbare bestanden: {len(bronnen)}")
    for b in bronnen:
        print(f"  - [{b.get('format','?')}] {b.get('name','?')}")
        print(f"    URL: {b.get('url','?')[:80]}...")

    # probeer de CSV te downloaden
    csvs = [b for b in bronnen if b.get('format','').upper() == 'CSV']
    if csvs:
        print(f"\n→ CSV gevonden, downloaden...")
        try:
            r = requests.get(csvs[0]['url'], timeout=30)
            from io import StringIO
            # probeer komma en puntkomma als scheidingsteken
            for sep in [';', ',']:
                try:
                    df_duo_live = pd.read_csv(StringIO(r.text), sep=sep, encoding='latin-1', nrows=5)
                    if len(df_duo_live.columns) > 2:
                        break
                except:
                    continue
            print(f"✅ CSV geladen — kolommen: {list(df_duo_live.columns)}")
            display(df_duo_live.head(3))
        except Exception as e:
            print(f"❌ CSV downloaden mislukt: {e}")

### 1.3 Conclusie data verzameling

Hieronder vatten we samen welke APIs werken en welke niet, en wat we doen als een API niet beschikbaar is.

In [ ]:
print("=" * 60)
print("CONCLUSIE DATA VERZAMELING")
print("=" * 60)
print()
print("CBS OData API (opendata.cbs.nl):")
if cbs_resultaat and 'value' in cbs_resultaat:
    print("  ✅ Bereikbaar en geeft data terug")
    print("  ⚠️ Wijk-codes in v3 (bijv. WK0363AA) zijn ANDERS dan in onze nooddata")
    print("     → We gebruiken nooddata zodat alles consistent is")
else:
    print("  ❌ Niet bereikbaar")
    print("  → We gebruiken nooddata (gebaseerd op echte CBS-cijfers 2023)")

print()
print("DUO Open Onderwijsdata:")
if duo_resultaat:
    print("  ✅ API zelf bereikbaar")
    print("  ⚠️ CSV-formaat is veranderd: geen PLAATSNAAM/wijk_code meer")
    print("  → We gebruiken nooddata (gebaseerd op echte DUO-verhoudingen 2018-2024)")
else:
    print("  ❌ Niet bereikbaar")
    print("  → We gebruiken nooddata")

---
## 2. DATA VERKENNING (EDA)

Nu laden we de data die het dashboard gebruikt en bekijken we die grondig.

**EDA = Exploratory Data Analysis.** Dat betekent:
- Hoe groot is de dataset? (rijen × kolommen)
- Welke kolommen zijn er en wat zijn de datatypes?
- Zijn er ontbrekende waarden?
- Hoe zijn de waarden verdeeld? (minimum, maximum, gemiddelde)
- Zijn er vreemde/verdachte waarden?

In [ ]:
import sys
sys.path.insert(0, r'c:\Users\AAchr\Documents\Casus 2 phyton minor\ROC')
from data import laad_alle_data, ADVIES_TYPEN, ADVIES_KLEUREN, SCHOOLJAREN

wijken_df, duo_df, bronnen = laad_alle_data()

print("Data geladen!")
print()
print("Gebruikte databronnen:")
for naam, bron in bronnen.items():
    status = '✅ Live' if 'Live' in bron else '📦 Eigen data'
    print(f"  {status} — {naam}")
    print(f"           {bron}")

### 2.1 Dataset 1: Wijken van Amsterdam (CBS-data)

Deze dataset bevat sociaaleconomische kenmerken per wijk.
Denk aan: gemiddeld inkomen, % niet-westers, WOZ-waarde, etc.

In [ ]:
print("AFMETINGEN:")
print(f"  Aantal wijken (rijen): {wijken_df.shape[0]}")
print(f"  Aantal kolommen:       {wijken_df.shape[1]}")
print()

# de voor ons relevante kolommen
relevante_kolommen = [
    'wijk_code', 'wijk_naam', 'stadsdeel',
    'gem_inkomen', 'pct_niet_westers', 'pct_uitkering',
    'pct_hoog_opgeleid', 'pct_laag_opgeleid', 'gem_woz',
    'pct_hoog_advies', 'pct_laag_advies', 'lat', 'lon'
]
bestaande_kolommen = [k for k in relevante_kolommen if k in wijken_df.columns]

print("EERSTE 5 RIJEN (relevante kolommen):")
display(wijken_df[bestaande_kolommen].head())

In [ ]:
print("DATATYPES per kolom:")
print(wijken_df[bestaande_kolommen].dtypes.to_string())

In [ ]:
print("ONTBREKENDE WAARDEN:")
ontbrekend = wijken_df[bestaande_kolommen].isnull().sum()
for kolom, n in ontbrekend.items():
    status = '✅' if n == 0 else f'⚠️  {n} ontbrekend'
    print(f"  {kolom:<25} {status}")

In [ ]:
print("BESCHRIJVENDE STATISTIEKEN:")
numeriek = [k for k in bestaande_kolommen
            if k not in ('wijk_code', 'wijk_naam', 'stadsdeel') and wijken_df[k].dtype in ('float64','int64')]
display(wijken_df[numeriek].describe().round(2))

In [ ]:
# Controleer op verdachte waarden (buiten verwacht bereik)
print("VALIDATIE — zijn de waarden realistisch?")
checks = [
    ('gem_inkomen',      15,  80,  'x€1.000 per jaar'),
    ('pct_niet_westers',  0, 100,  '%'),
    ('pct_uitkering',     0,  50,  '%'),
    ('pct_hoog_opgeleid', 0, 100,  '%'),
    ('pct_laag_opgeleid', 0, 100,  '%'),
    ('gem_woz',         100, 2000, 'x€1.000'),
    ('pct_hoog_advies',   0, 100,  '%'),
    ('pct_laag_advies',   0, 100,  '%'),
]
for kolom, min_v, max_v, eenheid in checks:
    if kolom not in wijken_df.columns:
        print(f"  ⚠️  {kolom}: kolom ONTBREEKT")
        continue
    reeks = wijken_df[kolom].dropna()
    te_laag  = (reeks < min_v).sum()
    te_hoog  = (reeks > max_v).sum()
    if te_laag == 0 and te_hoog == 0:
        print(f"  ✅ {kolom:<25} min={reeks.min():.1f}  max={reeks.max():.1f}  ({eenheid})")
    else:
        print(f"  ❌ {kolom:<25} {te_laag} te laag (<{min_v}), {te_hoog} te hoog (>{max_v})")

In [ ]:
# Verdeling per stadsdeel
print("WIJKEN PER STADSDEEL:")
print(wijken_df.groupby('stadsdeel')['wijk_naam'].apply(list).to_string())

In [ ]:
# Grafiek: inkomen per wijk gesorteerd
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# links: inkomen
gesorteerd = wijken_df.sort_values('gem_inkomen')
kleuren = ['#d73027' if s in ('Zuidoost','Nieuw-West') else '#1a9850' if s == 'Zuid' else '#4575b4'
           for s in gesorteerd['stadsdeel']]
axes[0].barh(gesorteerd['wijk_naam'], gesorteerd['gem_inkomen'], color=kleuren)
axes[0].set_xlabel('Gemiddeld inkomen (x€1.000 per jaar)')
axes[0].set_title('Gemiddeld inkomen per wijk')
axes[0].axvline(wijken_df['gem_inkomen'].mean(), color='black', linestyle='--', label='gemiddelde')
axes[0].legend()

# rechts: % hoog advies
gesorteerd2 = wijken_df.sort_values('pct_hoog_advies')
kleuren2 = ['#d73027' if s in ('Zuidoost','Nieuw-West') else '#1a9850' if s == 'Zuid' else '#4575b4'
            for s in gesorteerd2['stadsdeel']]
axes[1].barh(gesorteerd2['wijk_naam'], gesorteerd2['pct_hoog_advies'], color=kleuren2)
axes[1].set_xlabel('% hoog schooladvies (HAVO+)')
axes[1].set_title('% Hoog schooladvies per wijk')
axes[1].axvline(wijken_df['pct_hoog_advies'].mean(), color='black', linestyle='--', label='gemiddelde')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_inkomen_advies.png', dpi=120, bbox_inches='tight')
plt.show()
print("Opgeslagen als eda_inkomen_advies.png")

In [ ]:
# Correlatiematrix
corr_kolommen = [k for k in numeriek if k not in ('lat','lon')]
corr = wijken_df[corr_kolommen].corr().round(2)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            fmt='.2f', linewidths=0.5, ax=ax)
ax.set_title('Correlatiematrix — samenhang tussen wijkkenmerken', fontsize=13)
plt.tight_layout()
plt.savefig('eda_correlatiematrix.png', dpi=120, bbox_inches='tight')
plt.show()
print("\nCorrelatie met pct_hoog_advies:")
print(corr['pct_hoog_advies'].drop('pct_hoog_advies').sort_values(ascending=False).to_string())

### 2.2 Dataset 2: DUO Schooladviezen

Deze dataset bevat per school, per schooljaar en per adviestype hoeveel leerlingen dat advies kregen.

In [ ]:
print("AFMETINGEN:")
print(f"  Aantal rijen:    {duo_df.shape[0]:,}")
print(f"  Aantal kolommen: {duo_df.shape[1]}")
print()
print("KOLOMMEN EN DATATYPES:")
print(duo_df.dtypes.to_string())
print()
print("EERSTE 5 RIJEN:")
display(duo_df.head())

In [ ]:
print("ONTBREKENDE WAARDEN:")
ontbrekend2 = duo_df.isnull().sum()
for kolom, n in ontbrekend2.items():
    status = '✅' if n == 0 else f'⚠️  {n} ontbrekend ({n/len(duo_df)*100:.1f}%)'
    print(f"  {kolom:<25} {status}")

In [ ]:
print("UNIEKE WAARDEN PER CATEGORISCHE KOLOM:")
print(f"  Scholen (brin):       {duo_df['brin'].nunique()}")
print(f"  Wijken:               {duo_df['wijk_naam'].nunique()}")
print(f"  Stadsdelen:           {duo_df['stadsdeel'].nunique()} → {sorted(duo_df['stadsdeel'].unique())}")
print(f"  Schooljaren:          {duo_df['schooljaar'].nunique()} → {sorted(duo_df['schooljaar'].unique())}")
print(f"  Adviestypen:          {duo_df['advies_type'].nunique()} → {duo_df['advies_type'].unique().tolist()}")
print()
print("BESCHRIJVENDE STATISTIEKEN (numeriek):")
display(duo_df[['aantal_leerlingen','bijgesteld_hoger','pct','pct_bijgesteld']].describe().round(2))

In [ ]:
# Zijn alle adviestypen aanwezig in elk schooljaar?
print("VOLLEDIGHEIDSCHECK — adviestypen × schooljaren:")
kruistabel = duo_df.groupby(['schooljaar','advies_type'])['aantal_leerlingen'].sum().unstack(fill_value=0)
display(kruistabel)

In [ ]:
# Negatieve waarden of nul-leerlingen?
print("VALIDATIE DUO-data:")
negatief = (duo_df['aantal_leerlingen'] < 0).sum()
nul      = (duo_df['aantal_leerlingen'] == 0).sum()
print(f"  Rijen met negatief aantal_leerlingen: {negatief} {'✅' if negatief==0 else '❌'}")
print(f"  Rijen met nul leerlingen:             {nul}")
print(f"  Minimaal aantal leerlingen:           {duo_df['aantal_leerlingen'].min()}")
print(f"  Maximaal aantal leerlingen:           {duo_df['aantal_leerlingen'].max()}")
print()
# pct buiten 0-100?
pct_fout = ((duo_df['pct'] < 0) | (duo_df['pct'] > 100)).sum()
print(f"  Rijen met pct buiten 0-100%:          {pct_fout} {'✅' if pct_fout==0 else '❌'}")
print()
# Check of de percentages per wijk×jaar optellen tot ~100%
controle = duo_df.groupby(['wijk_naam','schooljaar'])['pct'].sum().round(0)
afwijkend = ((controle < 95) | (controle > 105)).sum()
print(f"  Wijk×jaar combinaties waarbij pct niet ~100% is: {afwijkend} {'✅' if afwijkend==0 else '⚠️'}")

In [ ]:
# Grafiek: adviesverdeling door de jaren heen (Amsterdam totaal)
amsterdam_trend = (duo_df.groupby(['schooljaar','advies_type'])['pct']
                   .mean().reset_index())
amsterdam_trend['schooljaar_nr'] = amsterdam_trend['schooljaar'].map(
    {j: i for i, j in enumerate(sorted(amsterdam_trend['schooljaar'].unique()))})

KLEUREN = {
    'Praktijkonderwijs': '#d73027', 'VMBO-BBL': '#f46d43', 'VMBO-KBL': '#fdae61',
    'VMBO-TL': '#fee08b', 'VMBO-TL/HAVO': '#d9ef8b', 'HAVO': '#66bd63',
    'HAVO/VWO': '#1a9850', 'VWO': '#006837'
}

fig, ax = plt.subplots(figsize=(12, 6))
for advies in ADVIES_TYPEN:
    subset = amsterdam_trend[amsterdam_trend['advies_type'] == advies]
    ax.plot(subset['schooljaar'], subset['pct'], marker='o',
            label=advies, color=KLEUREN.get(advies, 'gray'), linewidth=2)

ax.axvline('2023-2024', color='black', linestyle='--', alpha=0.5, label='← doorstroomtoets')
ax.set_xlabel('Schooljaar')
ax.set_ylabel('Gemiddeld % leerlingen')
ax.set_title('Adviesverdeling Amsterdam — trend 2018-2024')
ax.legend(loc='upper left', fontsize=9)
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('eda_adviestrend.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Effect doorstroomtoets: % bijgesteld advies per wijk
bijstelling = (duo_df[duo_df['schooljaar'] == '2023-2024']
               .groupby('wijk_naam', as_index=False)
               .agg(totaal=('aantal_leerlingen','sum'), bijgesteld=('bijgesteld_hoger','sum')))
bijstelling['pct_bij'] = (bijstelling['bijgesteld'] / bijstelling['totaal'] * 100).round(1)
bijstelling = bijstelling.merge(wijken_df[['wijk_naam','gem_inkomen','stadsdeel']], on='wijk_naam', how='left')
bijstelling = bijstelling.sort_values('pct_bij', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
kleuren = ['#d73027' if inc < 25 else '#fdae61' if inc < 35 else '#1a9850'
           for inc in bijstelling['gem_inkomen']]
ax.bar(bijstelling['wijk_naam'], bijstelling['pct_bij'], color=kleuren)
ax.set_xlabel('Wijk')
ax.set_ylabel('% leerlingen met hoger bijgesteld advies')
ax.set_title('Doorstroomtoets 2023-2024: % bijgesteld per wijk\n(rood = laag inkomen, groen = hoog inkomen)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('eda_bijstelling.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 3. DATAKWALITEIT — Eindrapport

Hier vatten we alle bevindingen samen.

In [ ]:
print("=" * 65)
print("EINDRAPPORT DATAKWALITEIT")
print("=" * 65)

print()
print("━━ DATASET 1: Wijken Amsterdam (CBS) ━━")
print(f"   Rijen:            {wijken_df.shape[0]} (= 22 Amsterdamse wijken)")
print(f"   Kolommen totaal:  {wijken_df.shape[1]}")
print(f"   Relevante kol.:   {len(bestaande_kolommen)}")
n_miss = wijken_df[bestaande_kolommen].isnull().sum().sum()
print(f"   Ontbrekend:       {n_miss} waarden {'✅' if n_miss == 0 else '⚠️'}")
print(f"   Bron:             Eigen data op basis van echte CBS-cijfers 2023")
print(f"   Status:           Schoon, volledig, klaar voor analyse")

print()
print("━━ DATASET 2: DUO Schooladviezen ━━")
print(f"   Rijen:            {duo_df.shape[0]:,} (school × jaar × adviestype)")
print(f"   Scholen:          {duo_df['brin'].nunique()}")
print(f"   Wijken:           {duo_df['wijk_naam'].nunique()}")
print(f"   Schooljaren:      {duo_df['schooljaar'].nunique()} ({SCHOOLJAREN[0]} t/m {SCHOOLJAREN[-1]})")
n_miss2 = duo_df.isnull().sum().sum()
print(f"   Ontbrekend:       {n_miss2} waarden {'✅' if n_miss2 == 0 else '⚠️'}")
print(f"   Bron:             Synthetische data, verhoudingen gebaseerd op DUO 2018-2024")
print(f"   Status:           Schoon, volledig, klaar voor analyse")

print()
print("━━ OPSCHONING UITGEVOERD ━━")
stappen = [
    "CBS API URL bijgewerkt (v4 → v3, odata4.cbs.nl → opendata.cbs.nl)",
    "Dubbele kolomnamen voorkomen (anti-duplicaat logica toegevoegd)",
    "Incompatibele CBS wijk-codes gedetecteerd → nooddata gebruikt",
    "Absolute aantallen omgezet naar percentages waar nodig",
    "Ontbrekende waarden (wijk_naam, lat, lon) aangevuld via merge",
    "Kapotte dependency verwijderd uit requirements.txt",
]
for s in stappen:
    print(f"   ✅ {s}")

print()
print("━━ CONCLUSIE ━━")
print("   Beide datasets zijn volledig en bevatten geen ontbrekende waarden.")
print("   De data is gevalideerd en klaar voor gebruik in het dashboard.")
print("=" * 65)

---
## 4. Opgeslagen grafieken

De volgende grafieken zijn aangemaakt en opgeslagen als PNG:

| Bestand | Inhoud |
|---------|--------|
| `eda_inkomen_advies.png` | Inkomen en % hoog advies per wijk |
| `eda_correlatiematrix.png` | Samenhang tussen alle wijkkenmerken |
| `eda_adviestrend.png` | Trend in adviezen 2018–2024 |
| `eda_bijstelling.png` | Effect doorstroomtoets per wijk |